# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jaivarshini-06/flyrank-ml-internship-jaivarshini/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

This rule uses two signals: staleness and search volume. Content that has not been updated for at least 180 days and has at least 300 impressions in the last 90 days will be prioritized for refresh review.

The reason code will be `stale_visible` when both conditions are met, and `below_rule_threshold` otherwise. The action will be `refresh_review` or `monitor`.

I will inspect how both signals relate to the observed decline indicator before applying the rule. The indicator is used only for the signal audit, not for scoring.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd

# Locate the repository and dataset
repo_dir = Path.cwd()

while repo_dir != repo_dir.parent and not (
    repo_dir / "data" / "raw" / "content_refresh_anonymized.csv"
).exists():
    repo_dir = repo_dir.parent

data_path = repo_dir / "data" / "raw" / "content_refresh_anonymized.csv"

if not data_path.exists():
    repo_dir = Path("/content/flyrank-ml-internship-jaivarshini")

    if not data_path.exists():
        subprocess.run([
            "git", "clone",
            "https://github.com/Jaivarshini-06/flyrank-ml-internship-jaivarshini.git",
            str(repo_dir)
        ], check=True)

    data_path = repo_dir / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

required = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
)
df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
)

# Used only for signal auditing, NOT scoring
df["audit_declining"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

# SIGNAL 1: STALENESS
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 90, 180, np.inf],
    labels=["0-90 days", "91-180 days", "181+ days"]
)

stale_table = df.groupby(
    "staleness_bucket", observed=False
).agg(
    n=("audit_declining", "size"),
    declining=("audit_declining", "sum"),
    decline_rate=("audit_declining", "mean")
)

stale_table["decline_rate_pct"] = (
    stale_table["decline_rate"] * 100
).round(2)

print("\nSIGNAL 1: STALENESS")
display(stale_table[["n", "declining", "decline_rate_pct"]])

rates = stale_table["decline_rate"].dropna()

if len(rates) < 2:
    stale_verdict = "FALSE"
    stale_explanation = "Not enough populated buckets to compare."
elif rates.iloc[-1] - rates.iloc[0] > 0.02:
    stale_verdict = "CONFIRMED"
    stale_explanation = "The oldest bucket has a higher decline rate."
elif rates.iloc[-1] - rates.iloc[0] < -0.02:
    stale_verdict = "OPPOSITE"
    stale_explanation = "The oldest bucket has a lower decline rate."
else:
    stale_verdict = "MIXED"
    stale_explanation = "The endpoint bucket rates are similar."

print("Verdict:", stale_verdict, "-", stale_explanation)


# SIGNAL 2: SEARCH VOLUME
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 99, 999, np.inf],
    labels=["Below 100", "100-999", "1000+"]
)

volume_table = df.groupby(
    "volume_bucket", observed=False
).agg(
    n=("audit_declining", "size"),
    declining=("audit_declining", "sum"),
    decline_rate=("audit_declining", "mean"),
    median_impressions=("impressions_90d", "median")
)

volume_table["decline_rate_pct"] = (
    volume_table["decline_rate"] * 100
).round(2)

print("\nSIGNAL 2: SEARCH VOLUME")
display(volume_table[
    ["n", "declining", "median_impressions", "decline_rate_pct"]
])

rates = volume_table["decline_rate"].dropna()

if len(rates) < 2:
    volume_verdict = "FALSE"
    volume_explanation = "Not enough populated buckets to compare."
elif rates.iloc[-1] - rates.iloc[0] > 0.02:
    volume_verdict = "CONFIRMED"
    volume_explanation = "The highest-volume bucket has a higher decline rate."
elif rates.iloc[-1] - rates.iloc[0] < -0.02:
    volume_verdict = "OPPOSITE"
    volume_explanation = "The highest-volume bucket has a lower decline rate."
else:
    volume_verdict = "MIXED"
    volume_explanation = "The endpoint bucket rates are similar."

print("Verdict:", volume_verdict, "-", volume_explanation)

Dataset shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

SIGNAL 1: STALENESS


,n,declining,decline_rate_pct
staleness_bucket,,,
0-90 days,20655,10576,51.20
91-180 days,9171,5604,61.11
181+ days,174,82,47.13


Verdict: OPPOSITE - The oldest bucket has a lower decline rate.

SIGNAL 2: SEARCH VOLUME


,n,declining,median_impressions,decline_rate_pct
volume_bucket,,,,
Below 100,7994,3110,12.0,38.90
100-999,8494,5121,388.0,60.29
1000+,13512,8031,4304.0,59.44


Verdict: CONFIRMED - The highest-volume bucket has a higher decline rate.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# One rule: stale content + at least 300 impressions
df["is_stale"] = (
    df["days_since_last_update"] >= 180
).fillna(False)

df["has_volume"] = (
    df["impressions_90d"] >= 300
).fillna(False)

qualifies = df["is_stale"] & df["has_volume"]

# Score: qualifying pages are ranked by their impressions
df["score"] = np.where(
    qualifies,
    df["impressions_90d"].fillna(0),
    0
)

# Exactly one reason code and one action label
df["reason_code"] = np.where(
    qualifies,
    "stale_visible",
    "below_rule_threshold"
)

df["action_label"] = np.where(
    qualifies,
    "refresh_review",
    "monitor"
)

# Rank all items
df = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False],
    na_position="last"
).reset_index(drop=True)

df["rank"] = range(1, len(df) + 1)

# Save output
output_dir = repo_dir / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d"
]

df[columns].to_csv(output_path, index=False)

print("Total rows:", len(df))
print("Refresh-review candidates:", int(qualifies.sum()))
print("CSV saved to:", output_path)

display(df[columns].head(10))


Total rows: 30000
Refresh-review candidates: 22
CSV saved to: /content/flyrank-ml-internship-jaivarshini/work/outputs/baseline_action_score.csv


,rank,content_id,score,reason_code,action_label,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,61678,stale_visible,refresh_review,194,61678
1,2,content_7368877ea310,59472,stale_visible,refresh_review,194,59472
2,3,content_1bfaa38ff26c,25715,stale_visible,refresh_review,194,25715
3,4,content_0a91db491d14,13299,stale_visible,refresh_review,193,13299
4,5,content_5feee3994adb,7812,stale_visible,refresh_review,194,7812
5,6,content_c2d929d83eaa,7558,stale_visible,refresh_review,193,7558
6,7,content_b16bd7307b39,4590,stale_visible,refresh_review,194,4590
7,8,content_fe16a55cd13d,4556,stale_visible,refresh_review,194,4556
8,9,content_ecb6215e79fd,4429,stale_visible,refresh_review,194,4429
9,10,content_928af3e22c80,1697,stale_visible,refresh_review,193,1697


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df.head(20)

for _, row in top20.iterrows():

    if row["action_label"] == "refresh_review":
        why = (
            f"{row['days_since_last_update']:.0f} days since update "
            f"and {row['impressions_90d']:,.0f} impressions."
        )

        confidence = "Moderate: both thresholds are met, but the rule is unvalidated."

        wrong = (
            "The page may already be accurate, or refreshing it may not "
            "improve its performance."
        )

    else:
        why = "The item does not meet both rule thresholds."
        confidence = "Low priority under this rule, not necessarily a healthy page."
        wrong = (
            "The page might still need urgent work despite missing a threshold."
        )

    print(f"\nRank: {int(row['rank'])}")
    print("Content ID:", row["content_id"])
    print("Action:", row["action_label"])
    print("Reason code:", row["reason_code"])
    print("Why it is here:", why)
    print("Confidence:", confidence)
    print("What would make it wrong:", wrong)
    print("-" * 60)



Rank: 1
Content ID: content_cf56e2e2e282
Action: refresh_review
Reason code: stale_visible
Why it is here: 194 days since update and 61,678 impressions.
Confidence: Moderate: both thresholds are met, but the rule is unvalidated.
What would make it wrong: The page may already be accurate, or refreshing it may not improve its performance.
------------------------------------------------------------

Rank: 2
Content ID: content_7368877ea310
Action: refresh_review
Reason code: stale_visible
Why it is here: 194 days since update and 59,472 impressions.
Confidence: Moderate: both thresholds are met, but the rule is unvalidated.
What would make it wrong: The page may already be accurate, or refreshing it may not improve its performance.
------------------------------------------------------------

Rank: 3
Content ID: content_1bfaa38ff26c
Action: refresh_review
Reason code: stale_visible
Why it is here: 194 days since update and 25,715 impressions.
Confidence: Moderate: both thresholds are me

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks: qualifying pages with the lowest impressions
flagged = df[df["action_label"] == "refresh_review"]

if len(flagged) > 0:
    weak_picks = flagged.sort_values(
        "impressions_90d", ascending=True
    ).head(5)

    print("Potential weak picks:")
    display(weak_picks[
        [
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "score",
            "reason_code",
            "action_label"
        ]
    ])
else:
    print("No pages met both thresholds. Review the audit tables.")

# Verify the score follows the stated rule
expected = np.where(
    (df["days_since_last_update"] >= 180).fillna(False)
    & (df["impressions_90d"] >= 300).fillna(False),
    df["impressions_90d"].fillna(0),
    0
)

assert np.allclose(df["score"], expected)

# Ensure the queue does not contain audit labels
assert "trend_direction" not in columns
assert "audit_declining" not in columns

assert output_path.exists()

print("Leakage check passed.")
print("Score uses only staleness and impressions.")
print("CSV exists:", output_path)

Potential weak picks:


,content_id,days_since_last_update,impressions_90d,score,reason_code,action_label
21,content_6476d1d8c050,313,304,304,stale_visible,refresh_review
20,content_4729b57ca036,301,335,335,stale_visible,refresh_review
19,content_ba00ffc6318c,211,345,345,stale_visible,refresh_review
18,content_b65fe2792b44,183,371,371,stale_visible,refresh_review
17,content_fd16e3475c29,183,429,429,stale_visible,refresh_review


Leakage check passed.
Score uses only staleness and impressions.
CSV exists: /content/flyrank-ml-internship-jaivarshini/work/outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.